# Démonstration – Diagnostic de défauts sur des pièces industrielles

Ce notebook présente visuellement le travail réalisé :

1. **La base de connaissances** : liens entre symptômes, défauts et composants
2. **Le module d'analyse des symptômes** : exemples et évaluation
3. **Les données d'images** : jeu NEU-DET, répartition et exemples
4. **Le module de détection visuelle** : entraînement, évaluation, prédictions et zones observées par le modèle

Exécuter toutes les cellules : bouton **Run All** en haut du notebook.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

# Placer le notebook à la racine du projet pour pouvoir importer le package « diagnostic »
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RACINE)
sys.path.insert(0, str(RACINE))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

from diagnostic.base_connaissances import BaseConnaissances

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", None)
kb = BaseConnaissances()
print(f"Base de connaissances : {len(kb.composants)} composants, {len(kb.symptomes)} symptômes, {len(kb.defauts)} défauts")

## 1. Base de connaissances

Chaque case indique à quel point un symptôme (ou un défaut visuel) pointe vers un composant, sur une échelle de 0 à 1.
Ces poids serviront au module de fusion pour classer les composants suspects.

In [ ]:
def carte_de_chaleur(relations, libelles_lignes, titre, ax):
    composants = list(kb.composants)
    matrice = np.array([[relations[l].get(c, 0) for c in composants] for l in relations])
    ax.imshow(matrice, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(composants)), [kb.composants[c]["libelle"] for c in composants], rotation=35, ha="right")
    ax.set_yticks(range(len(relations)), [libelles_lignes[l]["libelle"] for l in relations])
    for i in range(matrice.shape[0]):
        for j in range(matrice.shape[1]):
            if matrice[i, j] > 0:
                ax.text(j, i, f"{matrice[i, j]:.2f}".replace(".", ","), ha="center", va="center",
                        fontsize=8, color="white" if matrice[i, j] > 0.6 else "black")
    ax.set_title(titre)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={"width_ratios": [1.2, 1]})
carte_de_chaleur(kb.relations_symptomes, kb.symptomes, "Symptômes → composants", ax1)
carte_de_chaleur(kb.relations_defauts, kb.defauts, "Défauts visuels → composants", ax2)
plt.tight_layout()
plt.show()

Exemple : les causes et les vérifications recommandées pour un composant.

In [ ]:
composant = "roulement"
print(f"Composant : {kb.composants[composant]['libelle']}\n")
display(pd.DataFrame([
    {"Cause": c["libelle"],
     "Symptômes associés": ", ".join(kb.symptomes[s]["libelle"] for s in c["symptomes"]),
     "Défauts visuels associés": ", ".join(kb.defauts[d]["libelle"] for d in c["defauts"]) or "—"}
    for c in kb.composants[composant]["causes"]
]))
display(pd.DataFrame(kb.verifications(composant)).rename(columns={"priorite": "Priorité", "action": "Vérification"}))

## 2. Module d'analyse des symptômes

Le module lit une description libre en français et retourne les symptômes présents et ceux explicitement niés.
Il gère les conjugaisons, les fautes de frappe et les négations.

In [ ]:
from diagnostic.symptomes import ExtracteurSymptomes

extracteur = ExtracteurSymptomes(kb)
descriptions = [
    "La courroie siffle, patine et sent le caoutchouc brûlé.",
    "Ça ne vibre pas mais le roulement grince.",
    "Aucune fuite, par contre le moteur chauffe et le disjoncteur saute.",
    "Le réducteur fait un cliquetis à chaque rotation.",
    "La macine vibrre beaucoup.",
    "Ça brasse pas mal sur le banc d'essai.",
    "Le technicien a changé le filtre hier.",
]

lignes = []
for texte in descriptions:
    r = extracteur.extraire(texte)
    lignes.append({
        "Description": texte,
        "Symptômes détectés": ", ".join(f"{s['libelle']} ({s['confiance']:.0%})" for s in r["presents"]) or "—",
        "Symptômes niés": ", ".join(s["libelle"] for s in r["absents"]) or "—",
    })
pd.DataFrame(lignes)

**Essayez votre propre description** : modifiez le texte ci-dessous et exécutez la cellule.

In [ ]:
ma_description = "Le palier est bloqué, très chaud et il y a une flaque d'huile dessous."

r = extracteur.extraire(ma_description)
print("Symptômes détectés :", ", ".join(s["libelle"] for s in r["presents"]) or "aucun")
print("Symptômes niés     :", ", ".join(s["libelle"] for s in r["absents"]) or "aucun")

### Évaluation sur le corpus de développement (60 phrases annotées)

In [ ]:
from diagnostic.symptomes.evaluer import evaluer

resultats = evaluer(RACINE / "data" / "corpus_symptomes_dev.json", afficher_erreurs=False)

fig, ax = plt.subplots(figsize=(6, 3.5))
noms = ["Rappel", "Précision", "F1"]
valeurs = [resultats["rappel"] * 100, resultats["precision"] * 100, resultats["f1"] * 100]
barres = ax.bar(noms, valeurs, color=["#2b6cb0", "#4a90c2", "#7fb3d5"])
ax.axhline(85, color="crimson", linestyle="--", label="Critère visé (85 %)")
ax.bar_label(barres, labels=[f"{v:.1f} %".replace(".", ",") for v in valeurs], padding=3)
ax.set_ylim(0, 110)
ax.set_ylabel("%")
ax.set_title("Module symptômes – corpus de développement")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Données d'images : jeu NEU-DET

1 800 images en niveaux de gris (200 × 200 pixels) de défauts de surface sur de l'acier laminé, 300 par classe.

In [ ]:
from diagnostic.vision.donnees import DOSSIER_NEU

neu_vers_fr = {d["classe_neu"]: d["libelle"] for d in kb.defauts.values()}
classes = sorted(p.name for p in (DOSSIER_NEU / "train" / "images").iterdir() if p.is_dir())

lignes = []
for partie, nom in [("train", "Entraînement + validation"), ("validation", "Test")]:
    for c in classes:
        n = len(list((DOSSIER_NEU / partie / "images" / c).glob("*.jpg")))
        lignes.append({"Défaut": neu_vers_fr[c], "Partie": nom, "Images": n})
comptes = pd.DataFrame(lignes).pivot(index="Défaut", columns="Partie", values="Images")

ax = comptes.plot.bar(figsize=(8, 4), color=["#2b6cb0", "#e07b39"], rot=25)
ax.set_ylabel("Nombre d'images")
ax.set_xlabel("")
ax.set_title("Répartition des images par défaut")
plt.tight_layout()
plt.show()
comptes

In [ ]:
random.seed(0)
fig, axes = plt.subplots(len(classes), 5, figsize=(10, 13))
for i, c in enumerate(classes):
    fichiers = random.sample(sorted((DOSSIER_NEU / "train" / "images" / c).glob("*.jpg")), 5)
    for j, f in enumerate(fichiers):
        axes[i, j].imshow(Image.open(f).convert("L"), cmap="gray")
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
    axes[i, 0].set_ylabel(neu_vers_fr[c], fontsize=11, rotation=0, ha="right", va="center")
fig.suptitle("Exemples d'images pour chaque défaut", fontsize=14)
plt.tight_layout()
plt.show()

### Augmentation de données

Pendant l'entraînement, chaque image est transformée aléatoirement (retournements, rotation, luminosité, contraste) pour que le modèle généralise mieux.

In [ ]:
import torch
from diagnostic.vision.donnees import ECART_TYPE, MOYENNE, transformation_entrainement

def tenseur_vers_image(t):
    t = t * torch.tensor(ECART_TYPE).view(3, 1, 1) + torch.tensor(MOYENNE).view(3, 1, 1)
    return t.clamp(0, 1).permute(1, 2, 0).numpy()

exemple = sorted((DOSSIER_NEU / "train" / "images" / "scratches").glob("*.jpg"))[0]
image = Image.open(exemple)

torch.manual_seed(1)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
axes[0].imshow(image.convert("L"), cmap="gray")
axes[0].set_title("Originale")
for k, ax in enumerate(axes[1:], 1):
    ax.imshow(tenseur_vers_image(transformation_entrainement(image)))
    ax.set_title(f"Variante {k}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Module de détection visuelle

### Entraînement

ResNet18 préentraîné sur ImageNet, dernière couche remplacée par 6 sorties.
Si l'historique d'entraînement n'existe pas encore, la cellule suivante entraîne le modèle (environ 4 minutes).
Mettre `ENTRAINER = True` pour forcer un nouvel entraînement.

In [ ]:
from diagnostic.vision.entrainer import CHEMIN_HISTORIQUE, entrainer

ENTRAINER = False

if ENTRAINER or not CHEMIN_HISTORIQUE.exists():
    historique = entrainer(epoques=8)
else:
    historique = json.loads(CHEMIN_HISTORIQUE.read_text(encoding="utf-8"))

h = pd.DataFrame(historique)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(h["epoque"], h["perte"], marker="o")
ax1.set(title="Perte d'entraînement", xlabel="Époque", ylabel="Perte")
ax2.plot(h["epoque"], h["exactitude_validation"] * 100, marker="o", color="tab:green")
ax2.set(title="Exactitude en validation", xlabel="Époque", ylabel="%")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3)
    ax.set_xticks(h["epoque"])
plt.tight_layout()
plt.show()

### Évaluation sur le jeu de test (360 images jamais vues pendant l'entraînement)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from diagnostic.vision.donnees import charger_donnees
from diagnostic.vision.modele import charger

modele, classes_modele = charger()
_, _, chargeur_test, _ = charger_donnees()
noms_fr = [neu_vers_fr[c] for c in classes_modele]

vraies, predites, confiances = [], [], []
with torch.no_grad():
    for images, cibles in chargeur_test:
        probas = torch.softmax(modele(images), dim=1)
        confiances += probas.max(1).values.tolist()
        predites += probas.argmax(1).tolist()
        vraies += cibles.tolist()

rapport = pd.DataFrame(classification_report(vraies, predites, target_names=noms_fr, output_dict=True)).T
rapport = rapport.drop(index="accuracy").rename(columns={"precision": "Précision", "recall": "Rappel",
                                                          "f1-score": "F1", "support": "Images"})
rapport["Images"] = rapport["Images"].astype(int)
rapport.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(vraies, predites), display_labels=noms_fr).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set(title="Matrice de confusion – jeu de test", xlabel="Défaut prédit", ylabel="Défaut réel")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Exemples de prédictions

Titre vert : prédiction correcte. Titre rouge : erreur.

In [ ]:
random.seed(3)
indices = random.sample(range(len(vraies)), 12)
fig, axes = plt.subplots(3, 4, figsize=(12, 9.5))
for ax, i in zip(axes.flat, indices):
    chemin, _ = chargeur_test.dataset.samples[i]
    ax.imshow(Image.open(chemin).convert("L"), cmap="gray")
    correct = vraies[i] == predites[i]
    ax.set_title(f"Réel : {noms_fr[vraies[i]]}\nPrédit : {noms_fr[predites[i]]} ({confiances[i]:.0%})",
                 fontsize=9, color="green" if correct else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Où le modèle regarde-t-il ? (Grad-CAM)

La méthode Grad-CAM colore les zones de l'image qui ont le plus influencé la décision du modèle.
Elle permet de vérifier que le modèle s'appuie bien sur le défaut lui-même, et non sur un détail sans rapport.

In [ ]:
import torch.nn.functional as F
from diagnostic.vision.donnees import transformation_evaluation

def grad_cam(modele, tenseur):
    sauvegarde = {}

    def capturer(module, entree, sortie):
        sauvegarde["activations"] = sortie
        sortie.register_hook(lambda g: sauvegarde.__setitem__("gradients", g))

    crochet = modele.layer4.register_forward_hook(capturer)
    sortie = modele(tenseur)
    crochet.remove()
    classe = sortie.argmax(1).item()
    modele.zero_grad()
    sortie[0, classe].backward()

    poids = sauvegarde["gradients"].mean(dim=(2, 3), keepdim=True)
    carte = F.relu((poids * sauvegarde["activations"]).sum(dim=1, keepdim=True))
    carte = F.interpolate(carte, size=tenseur.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    carte = (carte / (carte.max() + 1e-8)).detach().numpy()
    return carte, classe

fig, axes = plt.subplots(2, len(classes_modele), figsize=(15, 5.5))
for j, c in enumerate(classes_modele):
    chemin = sorted((DOSSIER_NEU / "validation" / "images" / c).glob("*.jpg"))[0]
    image = Image.open(chemin)
    tenseur = transformation_evaluation(image).unsqueeze(0)
    carte, classe = grad_cam(modele, tenseur)
    gris = np.array(image.convert("L").resize((224, 224)))
    axes[0, j].imshow(gris, cmap="gray")
    axes[0, j].set_title(noms_fr[j], fontsize=10)
    axes[1, j].imshow(gris, cmap="gray")
    axes[1, j].imshow(carte, cmap="jet", alpha=0.45)
    axes[1, j].set_title(f"Prédit : {noms_fr[classe]}", fontsize=9)
    for ax in axes[:, j]:
        ax.axis("off")
fig.suptitle("Images de test (haut) et zones utilisées par le modèle (bas)", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Test sur des objets réels

Les photos sont placées dans `data/photos_reelles/<défaut attendu>/`, où le nom du dossier est
`microfissures`, `piqures`, `rayures`, `plaques`, `inclusions`, `calamine`, `aucun` (pièce sans défaut) ou `inconnu`.

Chaque photo est d'abord rapprochée du format NEU : niveaux de gris, carré central, contraste normalisé.
La ligne du haut montre la photo d'origine, celle du milieu l'image vue par le modèle, celle du bas la carte Grad-CAM.

In [ ]:
from diagnostic.vision.classifieur import ClassifieurDefauts
from diagnostic.vision.photos_reelles import lister_photos, preparer_photo

RECADRAGE = 0.8  # fraction centrale de la photo conservée (diminuer pour zoomer sur le défaut)

classifieur = ClassifieurDefauts()
photos = lister_photos()
print(f"{len(photos)} photo(s) trouvée(s)")

lignes = []
if photos:
    fig, axes = plt.subplots(3, len(photos), figsize=(3.2 * len(photos), 10), squeeze=False)
    for j, (chemin, attendu) in enumerate(photos):
        image = preparer_photo(chemin, RECADRAGE)
        resultats = classifieur.predire(image)
        tenseur = transformation_evaluation(image).unsqueeze(0)
        carte, _ = grad_cam(classifieur.modele, tenseur)
        gris = np.array(image.resize((224, 224)))

        axes[0, j].imshow(ImageOps.exif_transpose(Image.open(chemin)))
        axes[0, j].set_title(f"{chemin.name[:22]}\nAttendu : {attendu}", fontsize=9)
        axes[1, j].imshow(gris, cmap="gray")
        premier, second = resultats[0], resultats[1]
        bon = premier["id"] == attendu
        couleur = "green" if bon else ("red" if attendu in kb.defauts else "black")
        axes[1, j].set_title(f"{premier['libelle']} ({premier['probabilite']:.0%})\n"
                             f"2e : {second['libelle']} ({second['probabilite']:.0%})", fontsize=9, color=couleur)
        axes[2, j].imshow(gris, cmap="gray")
        axes[2, j].imshow(carte, cmap="jet", alpha=0.45)
        for ax in axes[:, j]:
            ax.axis("off")
        lignes.append({"Photo": chemin.name, "Attendu": attendu, "Prédit": premier["libelle"],
                       "Confiance": f"{premier['probabilite']:.0%}", "2e choix": second["libelle"]})
    plt.tight_layout()
    plt.show()
pd.DataFrame(lignes)

In [ ]:
avec_defaut = [l for l, (_, a) in zip(lignes, photos) if a in kb.defauts]
if avec_defaut:
    corrects = sum(kb.defauts[l["Attendu"]]["libelle"] == l["Prédit"] for l in avec_defaut)
    print(f"Photos de défauts correctement classées : {corrects}/{len(avec_defaut)} ({corrects / len(avec_defaut):.0%})")
sans_defaut = [l for l in lignes if l["Attendu"] == "aucun"]
if sans_defaut:
    print(f"Pièces sans défaut : {len(sans_defaut)}. Le modèle n'a pas de classe « normal », "
          "il leur attribue donc toujours un défaut.")

## Résumé

| Module | Critère visé | Résultat |
| --- | --- | --- |
| Détection visuelle | Rappel ≥ 90 %, précision ≥ 85 % | Voir la section 4 |
| Analyse des symptômes | ≥ 85 % des symptômes extraits | Voir la section 2 |

Prochaine étape : le module de fusion, qui combinera les deux sources pour classer les composants suspects.